In [1]:
def main(datasources, start_date, end_date):
    """
    因子构建主函数 - 纯VWAP偏离因子
    逻辑：收盘价低于当日VWAP → 日内被低估 → 因子值大 → 看多
    """
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]

    sql = f"""
    WITH cte_bar1m AS (
        SELECT
            date, instrument, volume,
            strftime(date, '%Y-%m-%d') AS trading_day,
            (ask_price1 + bid_price1) / 2.0 AS mid_price,
            mid_price * volume AS amount
        FROM {bar1m}
        WHERE ask_price1 > 0 AND bid_price1 > 0 AND volume > 0
    ),
    cte_daily AS (
        SELECT
            trading_day, instrument,
            SUM(amount) / NULLIF(SUM(volume), 0) AS vwap,
            LAST(mid_price ORDER BY date) AS close_price,
            COUNT(*) AS minute_count
        FROM cte_bar1m
        GROUP BY instrument, trading_day
    ),
    cte_raw AS (
        SELECT
            trading_day, instrument,
            (close_price - vwap) / (vwap + 1e-8) AS raw_deviation,
            minute_count
        FROM cte_daily
        WHERE minute_count > 10
    ),
    cte_cs AS (
        SELECT
            trading_day,
            AVG(raw_deviation) AS cs_mean,
            nanstd(raw_deviation) AS cs_std
        FROM cte_raw
        GROUP BY trading_day
    )
    SELECT
        CAST(r.trading_day AS DATETIME) AS date,
        r.instrument,
        -- 截面标准化后取反，再用tanh压缩到(-1,1)
        TANH((r.raw_deviation - c.cs_mean) / (c.cs_std + 1e-8) * -1.0) AS factor
        -- 变体1：去掉tanh压缩，直接用z-score
-- (r.raw_deviation - c.cs_mean) / (c.cs_std + 1e-8) * -1.0 AS factor
-- 变体3：偏离度用 (close-vwap)/close 而不是 (close-vwap)/vwap
-- 改 cte_raw 里这一行：(close_price - vwap) / (close_price + 1e-8) AS raw_deviation

    FROM cte_raw r
    JOIN cte_cs c ON r.trading_day = c.trading_day
    """

    df = dai.query(sql, filters={'date': [start_date, end_date]}, compression=True).df()

    # 对齐股票池
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]},
    ).df()
    stk_pool['date'] = pd.to_datetime(stk_pool['date'])
    df['date'] = pd.to_datetime(df['date'])

    df = pd.merge(df, stk_pool, how='inner', on=['date', 'instrument'])

    return df


if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()
    datasources = {'bar1m': 'bigalpha_2026_stock_bar1m'}
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    logger.info(f"计算纯VWAP偏离因子，区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    logger.info(f"读取因子库...")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )

[2026-07-19 16:20:35] [info     ] 计算纯VWAP偏离因子，区间：2024-01-01 00:00:00 ~ 2024-12-31 23:59:59
[2026-07-19 16:24:21] [info     ] 读取因子库...
[2026-07-19 16:24:21] [warning  ] bigalpha_eval._latest version='v4' (use ._latest for dev only, not for prod)
[2026-07-19 16:24:24] [info     ] bigalpha_eval.v4 开始运行 ..
[2026-07-19 16:24:24] [warning  ] 未传入官方评估窗口 start_date/end_date，回退到数据自身范围（仅建议本地调试时使用）
[2026-07-19 16:24:24] [info     ] 对齐中证1000历史成分后，官方评估窗口: 2024-01-02 至 2024-12-31
[2026-07-19 16:24:24] [info     ] ========== 数据检查 ==========
[2026-07-19 16:24:24] [info     ] 通过：列名检查（date/instrument + 至少 1 个因子列） factor_cols=['factor', 'close', 'volume', 'amount', 'turn', 'change_ratio', 'daily_return', 'momentum_5', 'reversal_5', 'volatility_5', 'total_market_cap', 'float_market_cap', 'pe_ttm', 'pb', 'ps_ttm', 'sma_20', 'ema_20', 'macd_diff_12_26_9', 'macd_dea_12_26_9', 'macd_hist_12_26_9', 'rsi_12', 'kdj_k_9_3_3', 'kdj_d_9_3_3', 'bias_20', 'cci_14', 'atr_14', 'roe_avg_ttm', 'roa_avg_ttm', 'gross_profit